# 00 · Profile, setup, budget, smoke test
**TERSE — *Is concision a skill or a character?***

One switch below selects a **profile**. `pilot` (~$8–12, 0.5B, A10G) answers: does the pipeline run, do the probes have headroom, does λ bite, what is the seed-noise floor?
`paper` is the template for the real study (several × the cost). Each profile has its **own Modal app, Volume, results folder, runs file and spec file**, so pilot numbers cannot leak into paper results.
Edit `lab/profiles.py` to change seeds / steps / models / tiers — nothing else needs touching.

Workflow: pilot → decide λ values, steps, seeds, probe difficulty → **write down the analysis plan** → switch to `paper`, redeploy (fresh Volume = fresh budget count).
Pilot runs are for design decisions only; do not report them as results.

In [ ]:
PROFILE = "pilot"            # <- "pilot" or "paper"; re-run the whole notebook after changing
open("profile.txt", "w").write(PROFILE)
%pip install -q modal numpy pandas matplotlib

**One-time auth (terminal):** `modal setup`. If your Modal plan offers a workspace spend limit, set it too — the ledger guard is a soft limit (parallel jobs each check before starting; killed jobs may not write a ledger row).

In [ ]:
import sys; sys.path.insert(0, "..")
import modal; print("modal", modal.__version__)

In [ ]:
!cd .. && TERSE_PROFILE={PROFILE} modal deploy modal_app.py

In [ ]:
import sys, json, os, glob, time; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lab import profiles, local, envs, collective, cost
P = profiles.get(); MAIN = P["main"]; HF = {t: v[0] for t, v in P["models"].items()}; FINAL = P["steps"]
SPEC = f"spec_{P['name']}.json"
print("PROFILE:", P["name"], "| GPU:", P["gpu"], "| cap $", P["cap"], "| main model:", HF[MAIN], "| steps:", FINAL)

In [ ]:
local.fn("warmup").remote([HF[t] for t in P["models"]])   # cache weights on this profile's Volume

### Smoke test: 3 GRPO steps per model → seconds/step → projected cost of the *whole plan*

In [ ]:
sps = {}
for tag in P["smoke"]:
    hf, mb = P["models"][tag]
    recs = local.fn("smoke").remote(dict(name=tag, model=hf, steps=3, micro_bs=mb, lam=0.5, lr=P["lr"]))
    sps[tag] = sum(r["sec"] for r in recs[1:]) / max(1, len(recs) - 1)          # skip step 0 (CUDA warm-up)
    print(tag, "s/step ≈", round(sps[tag], 1), "| generation share ≈", round(sum(r["t_gen"] for r in recs) / sum(r["sec"] for r in recs), 2), "| last:", recs[-1])
json.dump(sps, open(f"sec_per_step_{P['name']}.json", "w"))

In [ ]:
rows, tot = profiles.plan_cost(P, sps)
print("training cost by model ($):", {k: round(v, 1) for k, v in rows.items()})
print(f"PLAN TOTAL (train + ~25% evals/geometry/sampling) ≈ ${tot:.2f}  vs cap ${P['cap']:.0f}")
if tot > P["cap"]: print("!! plan exceeds cap — cut seeds/steps/models in lab/profiles.py before launching")
df, spent = local.ledger(); print(f"SPENT SO FAR: ${spent:.2f}")

**Decision rule.** Proceed only if the smoke test ran and the projected plan fits under the cap. For `paper`, if 1.5B s/step is > 90, cut `steps` or move to a cheaper-per-token setup (e.g. vLLM rollouts) before spending.